# End-of-Text Attender

Attends primarily to the beginning-of-sequence / end-of-text token

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
    show_type_tokens, show_type_filtered_tables,
    get_type_positions, TYPE_ID_TO_POSITION_KEY,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Matched Tokens

In [ ]:
show_type_tokens(str_tokens, "end_of_text")

## End-of-Text Attender — All 24 Heads

In [ ]:
tm = compute_all_type_metrics(cache, str_tokens)
ent_key = TYPE_ENTROPY_KEYS.get("end_of_text")
is_measurable = ("end_of_text", 0, 0) in tm
if is_measurable:
    results = sorted(
        [((l, h), tm[("end_of_text", l, h)]) for l in range(2) for h in range(12)],
        key=lambda x: x[1], reverse=True,
    )
    has_ent = ent_key and (ent_key, 0, 0) in tm
    if has_ent:
        lines = ["| Head | End-of-Text Attender % | Entropy % |", "|------|-------|-------|"]  
        for (l, h), pct in results:
            ent = tm[(ent_key, l, h)]
            lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    else:
        lines = ["| Head | End-of-Text Attender % |", "|------|-------|"]  
        for (l, h), pct in results:
            lines.append(f"| L{l}H{h} | {pct:.2f}% |")
    display(Markdown("\n".join(lines)))
else:
    print("No programmatic metric available for this type.")

## Top Heads

In [ ]:
pos_key = TYPE_ID_TO_POSITION_KEY.get("end_of_text")
_type_positions = get_type_positions(str_tokens).get(pos_key, []) if pos_key else []
sorted_heads = sorted(
    [((l, h), tm[("end_of_text", l, h)]) for l in range(2) for h in range(12) if ("end_of_text", l, h) in tm],
    key=lambda x: x[1], reverse=True,
)[:3]
for (l, h), pct in sorted_heads:
    ent_str = ""
    if ent_key and (ent_key, l, h) in tm:
        ent_str = f" | ent {tm[(ent_key, l, h)]:.2f}%"
    level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
    display(Markdown(f"---\n### L{l}H{h} — {pct:.2f}% ({level}){ent_str}"))
    show_head_pattern(str_tokens, cache, layer=l, head=h)
    attention = get_attention_pattern(cache, layer=l, head=h)
    if _type_positions:
        show_type_filtered_tables(str_tokens, attention, _type_positions, "End-of-Text Attender", top_k=15)
    show_attention_tables(str_tokens, attention, top_k=25)